# 03_Face Detection

## Objective
The objective of this notebook is to detect and isolate face regions from preprocessed images.

## Method
Face detection pipeline with cropped face extraction.

## Input
Preprocessed face images.

## Processing Tasks

1. Face Detection.
2. Detection Tasks.
3. Environment & Configuration.
4. Dataset Loading & Sample Selection.
5. Visual Inspection of Selected Images.
6. Initialize Face Detection Model.
7. Face Detection.
8. Bounding Box Visualization.
9. Face ROI Extraction.

## Output
- Detected/cropped face images.
- Detection quality visual examples.
- Face crops ready for feature extraction.

## Purpose
Extract valid face regions to improve feature quality in the next stage.


# 1. Face Detection

## Objective
The objective of this notebook is to detect and localize human faces
in the background-removed facial images.

## Input
Background-removed images from User_01 and User_02.

## 2. Detection Tasks
1. Load background-removed images.
2. Initialize the face detection algorithm.
3. Detect faces in each image.
4. Draw bounding boxes around detected faces.
5. Evaluate detection accuracy visually.

## Output
- Detected face locations.
- Bounding boxes around detected faces.
- Face regions ready for feature extraction.

## Purpose
Prepare facial images for Face ROI Extraction and Feature Extraction.


## 3. Environment & Configuration


In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Project Configuration
PROJECT_PATH = Path(
    r"E:\Multimodal Biometric Verification System"
)

# Input Dataset
INPUT_PATH = (
    PROJECT_PATH
    / "Dataset"
    / "Processed"
    / "Face"
    / "Face_Standardized_1024x1365"
)

# Output Dataset
OUTPUT_PATH = (
    PROJECT_PATH
    / "Dataset"
    / "Processed"
    / "Face"
    / "Face_Cropped"
)

# User Configuration
USERS = ["User_01", "User_02"]

# Testing Configuration
SAMPLES_PER_USER = 5
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Configuration completed successfully.")
print("Input Dataset Exists:", INPUT_PATH.exists())
print("Output Dataset Path:", OUTPUT_PATH)

## 4. Dataset Loading & Sample Selection


In [ ]:
# Load image paths for each user
image_paths = {}

for user in USERS:
    user_path = INPUT_PATH / user

    images = sorted([
        path for path in user_path.iterdir()
        if path.suffix.lower() in [".png", ".jpg", ".jpeg"]
    ])

    image_paths[user] = images

    print(f"{user}: {len(images)} images found.")


# Select random test samples
selected_images = {}

for user in USERS:
    if len(image_paths[user]) < SAMPLES_PER_USER:
        raise ValueError(
            f"Not enough images available for {user}."
        )

    selected_images[user] = random.sample(
        image_paths[user],
        SAMPLES_PER_USER
    )

    print(f"\n{user} - Selected Images:")

    for image_path in selected_images[user]:
        print(image_path.name)

## 5. Visual Inspection of Selected Images


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))

for row, user in enumerate(USERS):
    for col, image_path in enumerate(selected_images[user]):
        image = Image.open(image_path).convert("RGB")

        axes[row, col].imshow(image)
        axes[row, col].set_title(
            f"{user}\n{image_path.name}"
        )
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## 6. Initialize Face Detection Model

This section initializes the RetinaFace deep learning model, which will be used to detect and localize faces within the selected facial images.


In [ ]:
from retinaface import RetinaFace

print("RetinaFace imported successfully.")

## 7. Face Detection


In [ ]:
import os
import sys
from pathlib import Path

# Active Conda Environment
ENV_PATH = Path(sys.prefix)

# Set DeepFace model storage directory
DEEPFACE_HOME = ENV_PATH / ".deepface"

os.environ["DEEPFACE_HOME"] = str(DEEPFACE_HOME)

# Expected RetinaFace model path
MODEL_PATH = (
    DEEPFACE_HOME
    / "weights"
    / "retinaface.onnx"
)

print("Environment:", ENV_PATH)
print("Model Directory:", DEEPFACE_HOME)
print("Model Path:", MODEL_PATH)

In [ ]:
import os
import sys
from pathlib import Path

# Active Conda Environment
ENV_PATH = Path(sys.prefix)

# Configure model storage root
os.environ["DEEPFACE_HOME"] = str(ENV_PATH)

# Expected model location
MODEL_PATH = (
    ENV_PATH
    / ".deepface"
    / "weights"
    / "retinaface.onnx"
)

print("DEEPFACE_HOME:", os.environ["DEEPFACE_HOME"])
print("Expected Model Path:", MODEL_PATH)

In [ ]:
# Store detection results
detection_results = {}

for user in USERS:
    detection_results[user] = []

    for image_path in selected_images[user]:

        # Load image and convert RGBA to RGB
        image = Image.open(image_path).convert("RGB")

        # Convert RGB to BGR for RetinaFace
        image_array = np.array(image)[:, :, ::-1].copy()

        # Detect faces
        detections = RetinaFace.detect_faces(image_array)

        # Store results
        detection_results[user].append({
            "image_name": image_path.name,
            "detections": detections
        })

        # Display detection summary
        if isinstance(detections, dict):
            print(
                f"{image_path.name}: "
                f"{len(detections)} face(s) detected."
            )
        else:
            print(f"{image_path.name}: No face detected.")

## 8. Bounding Box Visualization


In [ ]:
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(2, 5, figsize=(20, 9))

for row, user in enumerate(USERS):

    for col, result in enumerate(detection_results[user]):

        image_path = INPUT_PATH / user / result["image_name"]
        image = Image.open(image_path).convert("RGB")

        ax = axes[row, col]
        ax.imshow(image)

        detections = result["detections"]

        if isinstance(detections, dict):

            for face_data in detections.values():

                x1, y1, x2, y2 = face_data["facial_area"]

                confidence = face_data["score"]

                # Draw bounding box
                rectangle = Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    linewidth=2,
                    edgecolor="lime",
                    facecolor="none"
                )

                ax.add_patch(rectangle)

                ax.set_title(
                    f"{result['image_name']}\n"
                    f"Confidence: {confidence:.3f}"
                )

        ax.axis("off")

plt.tight_layout()
plt.show()

## 9. Face ROI Extraction


In [ ]:
# Create output directories
for user in USERS:
    (OUTPUT_PATH / user).mkdir(
        parents=True,
        exist_ok=True
    )

# Cropping configuration
PADDING_RATIO = 0.15

print("Output directories created successfully.")

for user in USERS:
    print(user, ":", OUTPUT_PATH / user)

In [ ]:
# Extract face regions from detected bounding boxes

cropped_results = {}

for user in USERS:
    cropped_results[user] = []

    for result in detection_results[user]:

        image_path = INPUT_PATH / user / result["image_name"]
        image = Image.open(image_path).convert("RGB")

        image_width, image_height = image.size

        detections = result["detections"]

        if not isinstance(detections, dict):
            print(f"Skipping {result['image_name']}: No face detected.")
            continue

        for face_index, face_data in enumerate(detections.values()):

            x1, y1, x2, y2 = face_data["facial_area"]

            # Calculate padding
            box_width = x2 - x1
            box_height = y2 - y1

            pad_x = int(box_width * PADDING_RATIO)
            pad_y = int(box_height * PADDING_RATIO)

            # Expand bounding box
            crop_x1 = max(0, x1 - pad_x)
            crop_y1 = max(0, y1 - pad_y)
            crop_x2 = min(image_width, x2 + pad_x)
            crop_y2 = min(image_height, y2 + pad_y)

            # Crop face
            face_crop = image.crop((
                crop_x1,
                crop_y1,
                crop_x2,
                crop_y2
            ))

            # Save cropped image
            output_name = (
                f"{Path(result['image_name']).stem}"
                f"_Face_{face_index + 1}.png"
            )

            output_path = OUTPUT_PATH / user / output_name

            face_crop.save(output_path)

            cropped_results[user].append({
                "image_name": result["image_name"],
                "output_path": output_path,
                "bounding_box": (crop_x1, crop_y1, crop_x2, crop_y2),
                "confidence": face_data["score"],
                "face_size": face_crop.size
            })

            print(
                f"{user} | {output_name} | "
                f"Size: {face_crop.size}"
            )

print("\nFace cropping completed.")

In [ ]:
# Process the complete face dataset

cropped_results = {}

for user in USERS:

    cropped_results[user] = []

    for image_path in image_paths[user]:

        # Load standardized image
        image = Image.open(image_path).convert("RGB")

        image_width, image_height = image.size

        # Detect faces
        image_array = np.array(image)[:, :, ::-1].copy()
        detections = RetinaFace.detect_faces(image_array)

        if not isinstance(detections, dict):
            print(f"WARNING: No face detected in {image_path.name}")
            continue

        # Process detected faces
        for face_index, face_data in enumerate(detections.values()):

            x1, y1, x2, y2 = face_data["facial_area"]

            # Calculate padding
            box_width = x2 - x1
            box_height = y2 - y1

            pad_x = int(box_width * PADDING_RATIO)
            pad_y = int(box_height * PADDING_RATIO)

            # Expand bounding box within image boundaries
            crop_x1 = max(0, x1 - pad_x)
            crop_y1 = max(0, y1 - pad_y)
            crop_x2 = min(image_width, x2 + pad_x)
            crop_y2 = min(image_height, y2 + pad_y)

            # Crop face region
            face_crop = image.crop((
                crop_x1,
                crop_y1,
                crop_x2,
                crop_y2
            ))

            # Generate output filename
            output_name = (
                f"{image_path.stem}_Face_{face_index + 1}.png"
            )

            output_path = OUTPUT_PATH / user / output_name

            # Save cropped face
            face_crop.save(output_path)

            cropped_results[user].append({
                "image_name": image_path.name,
                "output_path": output_path,
                "bounding_box": (
                    crop_x1, crop_y1, crop_x2, crop_y2
                ),
                "confidence": face_data["score"],
                "face_size": face_crop.size
            })

    print(
        f"{user}: {len(cropped_results[user])} faces cropped "
        f"from {len(image_paths[user])} images."
    )

print("\nFull dataset face cropping completed.")